In [1]:
import tensorflow as tf
import kagglehub
import os
import numpy as np
import shutil
from sklearn.model_selection import train_test_split

print("⚡ Mendownload dataset...")
path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")

DATA_DIR = None
for root, dirs, files in os.walk(path):
    if os.path.basename(root).lower() == "color":
        DATA_DIR = root
        break

print("Dataset ada di:", DATA_DIR)

⚡ Mendownload dataset...
Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Dataset ada di: /kaggle/input/plantvillage-dataset/plantvillage dataset/color


In [2]:
TOMATO_DIR = "/content/tomato_only"

if os.path.exists(TOMATO_DIR):
    shutil.rmtree(TOMATO_DIR)
os.makedirs(TOMATO_DIR, exist_ok=True)

tomato_classes = [c for c in os.listdir(DATA_DIR) if c.startswith("Tomato")]
print(f"Ditemukan {len(tomato_classes)} kelas tomat:")
for c in tomato_classes:
    print(" -", c)

# Symlink folder biar hemat storage (tidak copy ulang gambar)
for c in tomato_classes:
    src = os.path.join(DATA_DIR, c)
    dst = os.path.join(TOMATO_DIR, c)
    os.symlink(src, dst)

print("\n✅ Folder tomat siap di:", TOMATO_DIR)

Ditemukan 10 kelas tomat:
 - Tomato___Late_blight
 - Tomato___healthy
 - Tomato___Early_blight
 - Tomato___Septoria_leaf_spot
 - Tomato___Tomato_Yellow_Leaf_Curl_Virus
 - Tomato___Bacterial_spot
 - Tomato___Target_Spot
 - Tomato___Tomato_mosaic_virus
 - Tomato___Leaf_Mold
 - Tomato___Spider_mites Two-spotted_spider_mite

✅ Folder tomat siap di: /content/tomato_only


In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 128

# Kumpulkan semua path gambar + labelnya
all_paths, all_labels = [], []
for idx, c in enumerate(sorted(tomato_classes)):
    folder = os.path.join(TOMATO_DIR, c)
    for fname in os.listdir(folder):
        all_paths.append(os.path.join(folder, fname))
        all_labels.append(idx)

class_names = sorted(tomato_classes)
print(f"Total gambar: {len(all_paths)}")

# Split: 70% train, 15% val, 15% test
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.3, stratify=all_labels, random_state=42
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def make_ds(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(train_paths, train_labels, shuffle=True)
val_ds = make_ds(val_paths, val_labels)
test_ds = make_ds(test_paths, test_labels)

Total gambar: 18160
Train: 12712 | Val: 2724 | Test: 2724


In [4]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=IMG_SIZE + (3,)),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
print("🚀 Training dimulai...")
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

model.save("/content/tomato_model.keras")
print("\n✅ Training selesai & model disimpan.")

🚀 Training dimulai...
Epoch 1/10
 21/100 ━━━━━━━━━━━━━━━━━━━━ 32s 410ms/step - accuracy: 0.3117 - loss: 2.0425

In [ ]:
# Cell ini sengaja dipisah biar testing tidak numpuk dengan proses training.
# Bisa di-run kapan saja setelah model sudah dilatih & disimpan.

loaded_model = tf.keras.models.load_model("/content/tomato_model.keras")

print("🧪 Evaluasi model di data TEST (belum pernah dilihat model)...")
test_loss, test_acc = loaded_model.evaluate(test_ds)

print("\n--- HASIL TESTING ---")
print(f"📊 Akurasi Test : {test_acc*100:.2f}%")
print(f"📉 Loss Test    : {test_loss:.4f}")